In [ ]:
from google.colab import drive
import os
import sys

# Monta il Drive
drive.mount('/content/drive')

# Definisci la cartella del progetto (modifica il percorso se diverso)
PROJECT_ROOT = '/content/drive/MyDrive/Speach_Emotion_Recognition'
os.chdir(PROJECT_ROOT)

# Aggiornare la cartuccia al sistema per consentire le importazioni (ad esempio da models.model import...)
sys.path.append(PROJECT_ROOT)

In [ ]:
!pip install -r requirements.txt

In [ ]:
# Controlla se i file esistono
test_path = 'dataset/augmentedDataset_9K_RandomSplit/X_train.npy'
if os.path.exists(test_path):
    print(" File trovati con successo su Drive!")
else:
    print(" Attenzione: percorso non trovato. Controlla la cartella su Drive.")

In [ ]:
# @title Configurazione dell'esperimento
esperimento = "Random Split (Augmented)" # @param ["Actor Split (Augmented)", "Random Split (Augmented)", "Pure Dataset"]

if esperimento == "Actor Split (Augmented)":
    data_path = 'dataset/augmentedDataset_9K_ActorSplit'
    results_folder = 'res_ActorSplit'
elif esperimento == "Random Split (Augmented)":
    data_path = 'dataset/augmentedDataset_9K_RandomSplit'
    results_folder = 'res_RandomSplit'
else:
    data_path = 'dataset/pureDataset_RandomSplit'
    results_folder = 'res_PureDataset'

results_path = os.path.join('checkpoints', results_folder)
os.makedirs(results_path, exist_ok=True)

In [ ]:
from train import run_training

# Avvia l'addestramento (i parametri possono essere letti dalle variabili sopra)
# augment=True se l'esperimento non è "Pure Dataset"
is_augment = False if experiment == "Pure Dataset" else True

history, model = run_training(
    data_path=data_path, 
    results_folder=results_folder, 
    augment=is_augment,
    epochs=60
)

In [ ]:
from utils.visuals import plot_training_history, plot_confusion_matrix
from sklearn.metrics import classification_report
from data.dataset import LABEL_MAP
import pandas as pd
import numpy as np
import os
import shutil

# - 1. SALVATAGGIO DI GRAFICI E LOG ---
# results_path è ora definito dinamicamente in base all'esperimento scelto
print(f" Salvataggio dei risultati in: {results_path}")

# Salva il grafico delle tendenze (Precisione/Perdita)
plot_training_history(history, save_path=os.path.join(results_path, 'history_plot.png'))

# Salva i registri numerici in CSV
hist_df = pd.DataFrame(history.history)
hist_df.to_csv(os.path.join(results_path, 'metrics_log.csv'), index=False)

# - 2. GESTIONE DEL MODELLO ---
# Utilizziamo il percorso dinamico del checkpoint definito in train.py (ad esempio checkpoint/res_ActorSplit/best_model.keras)
# Rispetto al tuo codice, il percorso ora riflette la cartella dell'esperimento
local_model = os.path.join('checkpoints', results_folder, 'best_model.keras')

if os.path.exists(local_model):
    shutil.copy(local_model, os.path.join(results_path, 'best_model_final.keras'))
    print(f" Miglior modello ({local_model}) copiato nella cartella dei risultati.")

# --- 3. VALUTAZIONE FINALE ---
print(" Avviare la valutazione finale sul set di test...")

# Carichiamo i dati di test (X_test e y_test)
X_test = np.load(os.path.join(data_path, 'X_test.npy'))
y_test = np.load(os.path.join(data_path, 'y_test.npy'))

# Generiamo previsioni
y_pred_probs = model.predict(X_test)
y_pred = np.argmax(y_pred_probs, axis=1)

# Convertire le etichette (stringhe) y_test in indici numerici utilizzando LABEL_MAP
# Gestiamo sia il caso in cui y_test è già numerico sia la stringa del caso
if isinstance(y_test[0], (str, np.str_)):
    y_true = np.array([LABEL_MAP[label] for label in y_test])
else:
    y_true = y_test

# Genera e salva la matrice di confusione
label_names = list(LABEL_MAP.keys())
plot_confusion_matrix(y_true, y_pred, labels=label_names, 
                      save_path=os.path.join(results_path, 'confusion_matrix.png'))

# Genera report di classificazione e salva (Precisione, Richiamo, F1)
report = classification_report(y_true, y_pred, target_names=label_names)
with open(os.path.join(results_path, 'classification_report.txt'), 'w') as f:
    f.write(f"Experiment: {results_folder}\n")
    f.write(f"Dataset: {data_path}\n")
    f.write("-" * 30 + "\n")
    f.write(report)

print(report)
print(" Valutazione completata e risultati salvati!")